# Chemprop-IR Data generation

In [ ]:
import os
import sys
from pathlib import Path

# Get the current working directory
notebook_dir = Path.cwd()

# Add the chemprop-IR directory to the path
chemprop_ir_path = notebook_dir / "chemprop-IR"
sys.path.append(str(chemprop_ir_path))

from argparse import Namespace
import csv
from typing import List, Optional
import numpy as np
import torch
from tqdm import tqdm

In [ ]:
from chemprop.train.predict import predict
from chemprop.data import MoleculeDataset
from chemprop.data.utils import get_data, get_data_from_smiles
from chemprop.utils import load_args, load_checkpoint, load_scalers
from chemprop.train.spectral_loss import roundrobin_sid
from chemprop.features import get_available_features_generators
get_available_features_generators()

In [ ]:
config_dict = {
    "gpu": list(range(torch.cuda.device_count())),
    "test_path": ["./models/chemprop-ir/ir_models_data/solvation_example/solvation_spectra.csv"],
    "use_compound_names": [False],
    "preds_path": ["./models/chemprop-ir/ir_models_data/ir_preds_test_2.csv"],
    "checkpoint_dir": ["./models/chemprop-ir/ir_models_data/experiment_model/model_files"],
    "spectra_type": ["experimental"],
    "spectra_type_nr": [0],
    "checkpoint_path": [None],
    "batch_size": [50],
    "no_cuda": [[False]],
    "features_generator": [None],
    "features_path": [None],
    "max_data_size": [100],
    "ensemble_variance": [False],
    "ensemble_variance_conv": [0.0],
}

from argparse import Namespace

def parse_arguments(hyperparameters):
    parsed_args = {key: val[0] for key, val in hyperparameters.items()}
    return Namespace(**parsed_args)

args = parse_arguments(config_dict)

In [ ]:
from chemprop.train import make_predictions
from chemprop.parsing import modify_predict_args

modify_predict_args(args)

### Simulate IR for big ZINC dataset

In [ ]:
import os
import sys
from pathlib import Path

# Get the current working directory (where the notebook is running)
notebook_dir = Path.cwd()

# Add the chemprop-IR directory to the path
sys.path.append(str(notebook_dir / "chemprop-IR"))

from argparse import Namespace
import csv
from typing import List, Optional

import numpy as np
import torch
from tqdm import tqdm

# Import chemprop modules
from chemprop.train.predict import predict
from chemprop.data import MoleculeDataset
from chemprop.data.utils import get_data, get_data_from_smiles
from chemprop.utils import load_args, load_checkpoint, load_scalers
from chemprop.train.spectral_loss import roundrobin_sid
from chemprop.features import get_available_features_generators
get_available_features_generators()

from chemprop.train import make_predictions
from chemprop.parsing import modify_predict_args
import pandas as pd
from rdkit import Chem

In [ ]:
config_dict = {
    "gpu": list(range(torch.cuda.device_count())),
    "test_path": ["./models/chemprop-ir/ir_models_data/solvation_example/solvation_spectra.csv"],
    "use_compound_names": [False],
    "preds_path": ["./models/chemprop-ir/ir_models_data/ir_preds_test_2.csv"],
    "checkpoint_dir": ["./models/chemprop-ir/ir_models_data/experiment_model/model_files"],
    "spectra_type": ["experimental"],
    "spectra_type_nr": [0],
    "checkpoint_path": [None],
    "batch_size": [50],
    "no_cuda": [[False]],
    "features_generator": [None],
    "features_path": [None],
    "max_data_size": [100],
    "ensemble_variance": [False],
    "ensemble_variance_conv": [0.0],
}



def parse_arguments(hyperparameters):
    parsed_args = {key: val[0] for key, val in hyperparameters.items()}
    return Namespace(**parsed_args)


# Function to generate spectral data from smiles (dummy function for demonstration)
def generate_spectral_data_batch(args, smiles):
    avg_preds, predictions_df = make_predictions(args, smiles=smiles_list)
    # TODO: Implement the neural network-based spectral data generation
    return avg_preds

def calculate_molecular_formulas(smiles_list: list) -> list:
    formulas = []
    for smiles in smiles_list:
        molecule = Chem.MolFromSmiles(smiles)
        formula = Chem.rdMolDescriptors.CalcMolFormula(molecule)
        formulas.append(formula)
    return formulas

args = parse_arguments(config_dict)
modify_predict_args(args)

In [ ]:
base_dir = os.path.dirname(os.path.abspath('__file__'))
csv_file = os.path.join(base_dir,"/data/IBM_dataset/IBM_SMI_data.csv")
df = pd.read_csv(csv_file)

# Create a directory to save the spectral data files
output_directory = os.path.join(base_dir,"/data/IBM_dataset/IBM_ir_sim")
if not os.path.exists(output_directory):
    os.makedirs(output_directory)

molecular_formulas = []
smiles_list = []
sample_id_list = []

batch_size = 64

# Step 2 and 3: Process in batches of 64
for i in tqdm(range(0, len(df), batch_size)):
    batch = df.iloc[i:i+batch_size]
    smiles_list = batch['SMILES'].tolist()
    sample_ids = batch['sample-id'].tolist()
    
    # Generate spectral data for the batch
    spectral_data_batch = generate_spectral_data_batch(args, smiles_list)
    
    
    # Save spectral data for each sample in the batch
    for j, spectral_data in enumerate(spectral_data_batch):
        output_file_path = os.path.join(output_directory, f"{sample_ids[j]}.csv")
        pd.DataFrame({'spectra': spectral_data}).to_csv(output_file_path, index=False)

        
    # Step 5: Calculate molecular formula and append to list
    formula = calculate_molecular_formulas(smiles_list)
    molecular_formulas.extend(formula)
    smiles_list.extend(smiles_list)
    sample_id_list.extend(sample_ids)

    
    

In [ ]:
# Save the accumulated data into a DataFrame
results_df = pd.DataFrame({
    'SMILES': smiles_list,
    'sample-id': sample_id_list,
    'formula': molecular_formulas
})
        
# Step 6: Save the updated DataFrame to a new CSV file
new_csv_path = os.path.join(base_dir,"/data/IBM_dataset/chemprop-ir-sim-IBM.csv")

results_df.to_csv(new_csv_path, index=False)


In [ ]:
import os
output_directory = os.path.join(base_dir,"/data/IBM_dataset/IBM_ir_sim")
len(os.listdir(output_directory))